# GPU check for paged-kv-cache

Runs the CUDA backend's tests on a real GPU. Each test compares the GPU kernels against `CpuBackend`, the reference implementation, element by element.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. The last cell prints the test results. The first run takes a few minutes, most of it compiling.

On Kaggle, turn on a GPU accelerator and internet access in the notebook settings first.

In [ ]:
# Set BRANCH to test a pull request before it is merged.
REPO = "https://github.com/SohamDutta02/paged-kv-cache"
BRANCH = "main"

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import glob, os, subprocess

!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":/usr/local/cuda/bin:" + os.environ["PATH"]

# cudarc loads NVRTC at runtime. Use the toolkit's copy if there is one, else install it.
nvrtc = sorted(glob.glob("/usr/local/cuda*/lib64/libnvrtc.so*"))
if nvrtc:
    lib = os.path.dirname(nvrtc[0])
else:
    subprocess.run(["pip", "install", "-q", "nvidia-cuda-nvrtc-cu12"], check=True)
    import nvidia.cuda_nvrtc
    lib = os.path.join(nvidia.cuda_nvrtc.__path__[0], "lib")
    if not os.path.exists(os.path.join(lib, "libnvrtc.so")):
        os.symlink("libnvrtc.so.12", os.path.join(lib, "libnvrtc.so"))
os.environ["LD_LIBRARY_PATH"] = lib + ":" + os.environ.get("LD_LIBRARY_PATH", "")
print("NVRTC from", lib)
!cargo --version

In [ ]:
!rm -rf paged-kv-cache && git clone -q --depth 1 --branch {BRANCH} {REPO}
!cd paged-kv-cache && git log -1 --format='Testing %h: %s'

In [ ]:
!cd paged-kv-cache && cargo test --release -p paged-kv-cuda --features cuda -- --test-threads=1 2>&1 | grep -vE '^\s+(Compiling|Downloaded|Downloading)'